# AI Stylist: train the garment cut-out model (FR-05)

Trains a small on-device model that outlines each garment in a photo, so the app can remove the
background from a flat-lay or hanger photo and cut each garment out of a photo of someone
wearing it. Same data and licence as the classifier (Fashionpedia, CC BY 4.0; P-3, D-08).

**Before you start:** *Runtime → Change runtime type → GPU* (T4 is enough).

**Time:** downloading and preparing the data takes about 45–90 minutes; training about
1.5–3 hours on a T4. Outputs are saved to Google Drive, so a disconnect loses at most the
current step. Run the **smoke test** (step 7) first: it catches problems in about 10 minutes
instead of after hours of training.

In [ ]:
# 1. Check the GPU and library versions.
!nvidia-smi -L || echo "No GPU: Runtime > Change runtime type > GPU"
import tensorflow as tf, keras, sys
print("Python", sys.version.split()[0], "| TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
# 2. Keep outputs on Google Drive (survives disconnects).
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/ai_stylist_training"
!mkdir -p {DRIVE}/annotations {DRIVE}/runs

In [ ]:
# 3. Get the training code. pycocotools decodes compressed masks if the files use them.
BRANCH = "feature/fr05-garment-segmentation"  # change to "main" once merged
!rm -rf /content/ai-stylist
!git clone --depth 1 --branch {BRANCH} https://github.com/amarhbaneen/ai-stylist.git /content/ai-stylist
!pip -q install pycocotools
%cd /content/ai-stylist/training
!python -m unittest discover -s tests

In [ ]:
# 4. Download annotations (cached on Drive since the classifier run) and images.
import os
BASE = "https://s3.amazonaws.com/ifashionist-dataset"
for name in ["instances_attributes_train2020.json", "instances_attributes_val2020.json"]:
    if not os.path.exists(f"{DRIVE}/annotations/{name}"):
        !wget -q --show-progress -O {DRIVE}/annotations/{name} {BASE}/annotations/{name}
!mkdir -p /content/data
!wget -q --show-progress -c -O /content/data/train2020.zip {BASE}/images/train2020.zip
!wget -q --show-progress -c -O /content/data/val_test2020.zip {BASE}/images/val_test2020.zip

In [ ]:
# 5. Unzip (prints the folder names; the classifier run used /content/data/train and /content/data/test).
!cd /content/data && unzip -q -o train2020.zip && unzip -q -o val_test2020.zip
!ls /content/data
TRAIN_IMAGES = "/content/data/train"
TEST_IMAGES = "/content/data/test"
ANN = f"{DRIVE}/annotations"

## Smoke test (about 10 minutes)

Runs every step on 300 photos with one short epoch. The numbers mean nothing; it only proves
the whole pipeline (masks → training → evaluation → export with the XNNPACK check) works on
this Colab before the long run. If it fails, send me the error.

In [ ]:
# 6. Prepare a small sample and check how the masks are stored.
SMOKE = "/content/seg_smoke"
!python -m ai_stylist_training.seg_prepare --annotations {ANN}/instances_attributes_train2020.json --images {TRAIN_IMAGES} --out {SMOKE} --split train --limit 300
!python -m ai_stylist_training.seg_prepare --annotations {ANN}/instances_attributes_val2020.json --images {TEST_IMAGES} --out {SMOKE} --split test --limit 100

In [ ]:
# 7. Train one short epoch per stage, evaluate and export.
SMOKE_RUN = "/content/seg_smoke_run"
!python -m ai_stylist_training.seg_train --data {SMOKE} --out {SMOKE_RUN} --head-epochs 1 --finetune-epochs 1
!python -m ai_stylist_training.seg_evaluate --data {SMOKE} --run {SMOKE_RUN}
!python -m ai_stylist_training.seg_export --data {SMOKE} --run {SMOKE_RUN} --check-samples 20 --calibration-samples 20; echo "export exit code: $?"

## Full run

Exit code 1 from the smoke export only means the barely trained model's variants disagree; any
Python traceback is a real problem. Then prepare all photos (the longest CPU step) and train.

In [ ]:
# 8. Prepare everything. Prints sample counts, mask formats and anything skipped.
DATA = "/content/seg"
!python -m ai_stylist_training.seg_prepare --annotations {ANN}/instances_attributes_train2020.json --images {TRAIN_IMAGES} --out {DATA} --split train --holdout 0.1 --synthetic 0.5
!python -m ai_stylist_training.seg_prepare --annotations {ANN}/instances_attributes_val2020.json --images {TEST_IMAGES} --out {DATA} --split test --synthetic 0.5
!df -h /content | tail -1

In [ ]:
# 9. Train (3 epochs decoder only, then up to 15 with the top of the backbone).
RUN = f"{DRIVE}/runs/seg1"
!python -m ai_stylist_training.seg_train --data {DATA} --out {RUN}

In [ ]:
# 10. Evaluate on the test photos (real and synthetic flat-lay separately).
!python -m ai_stylist_training.seg_evaluate --data {DATA} --run {RUN}

In [ ]:
# 11. Export to LiteRT and check the variants with XNNPACK.
!python -m ai_stylist_training.seg_export --data {DATA} --run {RUN} --model-id garment-seg --version 1.0.0
!ls -la {RUN}

In [ ]:
# 12. Download what the app needs (also kept in Drive under runs/seg1).
from google.colab import files
for name in ["garment-seg.tflite", "seg_labels.json", "seg_model_card.json", "seg_eval_report.json"]:
    files.download(f"{RUN}/{name}")

## Next

Send back `garment-seg.tflite`, `seg_labels.json`, `seg_model_card.json` and
`seg_eval_report.json`. They get checked (size, SHA-256, cut-out quality) before the model is
added to the app.